# 16 — Final Model Comparison

Combines only saved experiment metrics and prediction tables. Macro F1 and class-wise performance take priority over accuracy. Missing optional models remain explicitly unavailable rather than being assigned invented scores.

In [ ]:
import sys,json
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from src.post_05.common import METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR, ensure_output_dirs, new_artifact_path
ensure_output_dirs()
rows=[]
class_reports=[]
for path in sorted(METRICS_DIR.glob('*.json')):
    try: payload=json.loads(path.read_text())
    except (OSError,json.JSONDecodeError): continue
    if 'cycle' in payload and 'patient' in payload:
        for level in ('cycle','patient'):
            metric=payload[level]; rows.append({'model':path.stem.replace('_metrics',''),'input':'engineered features','level':level,'Accuracy':metric['accuracy'],'Macro Precision':metric['macro_precision'],'Macro Recall':metric['macro_recall'],'Macro F1':metric['macro_f1'],'Weighted F1':metric['weighted_f1']})
    elif 'metrics' in payload:
        metric=payload['metrics']; name=path.stem.replace('_',' '); rows.append({'model':name,'input':'Mel' if 'baseline' in path.stem else 'Mel + MFCC + chroma','level':'cycle','Accuracy':metric['accuracy'],'Macro Precision':metric['macro_precision'],'Macro Recall':metric['macro_recall'],'Macro F1':metric['macro_f1'],'Weighted F1':metric['weighted_f1']})
comparison=pd.DataFrame(rows).drop_duplicates(['model','level'])
if comparison.empty: print('No saved model metrics found. Run available experiments first.')
else:
    comparison=comparison.sort_values(['level','Macro F1','Weighted F1'],ascending=[True,False,False]); display(comparison)
    dest=new_artifact_path(METRICS_DIR/'final_model_comparison.csv'); comparison.to_csv(dest,index=False)
    cycle=comparison[comparison.level.eq('cycle')].sort_values('Macro F1',ascending=False)
    if not cycle.empty:
        ax=cycle.plot.bar(x='model',y=['Macro F1','Weighted F1','Accuracy'],figsize=(11,6)); ax.set_title('Held-out cycle model comparison'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(new_artifact_path(FIGURES_DIR/'16_model_comparison.png'),dpi=160); plt.show()
for path in sorted(PREDICTIONS_DIR.glob('*patient_predictions.csv')):
    frame=pd.read_csv(path)
    if {'diagnosis','prediction'}<=set(frame.columns):
        report=classification_report(frame.diagnosis.astype(str),frame.prediction.astype(str),output_dict=True,zero_division=0)
        class_reports.append(pd.DataFrame(report).T.assign(model=path.stem))
if class_reports:
    per_class=pd.concat(class_reports).reset_index(names='class'); display(per_class); dest=new_artifact_path(METRICS_DIR/'patient_per_class_comparison.csv'); per_class.to_csv(dest,index=False)
    model_path=sorted(PREDICTIONS_DIR.glob('*patient_predictions.csv'))[0]; frame=pd.read_csv(model_path); labels=sorted(set(frame.diagnosis.astype(str))|set(frame.prediction.astype(str)))
    sns.heatmap(confusion_matrix(frame.diagnosis.astype(str),frame.prediction.astype(str),labels=labels),annot=True,fmt='d',xticklabels=labels,yticklabels=labels); plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Patient-level confusion matrix: {model_path.stem}'); plt.tight_layout(); plt.savefig(new_artifact_path(FIGURES_DIR/'16_patient_confusion_matrix.png'),dpi=160); plt.show()
print('Select by macro F1, minority-class recall, and confusion-matrix review; do not select by accuracy alone. Some disease classes have too few patients for reliable patient-level estimates.')